<a href="https://colab.research.google.com/github/JUST-ZARI/hotel-clv-prediction/blob/main/notebooks/03_merge_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone repository
!git clone https://github.com/JUST-ZARI/hotel-clv-prediction.git
%cd /content/hotel-clv-prediction

Cloning into 'hotel-clv-prediction'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 103 (delta 45), reused 13 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 3.51 MiB | 3.65 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/hotel-clv-prediction


# Merge Dataset 1 + Dataset 2 into one unified DataFrame
Concatenates the two harmonized datasets.This is a union-columns only present in one source get NaN for rows from the other source.

In [2]:
import pandas as pd

# Show all columns when displaying the DataFrames
pd.set_option('display.max_columns', None)

# Load Dataset 1 and Dataset 2 from the CSV file
df1 = pd.read_csv("data/processed/dataset1_harmonized.csv")
df2 = pd.read_csv("data/processed/dataset2_harmonized.csv")

# Print the number of rows and columns in each dataset
print("Dataset 1:", df1.shape)
print("Dataset 2:", df2.shape)

Dataset 1: (119390, 32)
Dataset 2: (36275, 20)


#Check which columns both datasets have in common before combining them.


In [3]:
# Find columns that are present in both datasets
shared = set(df1.columns) & set(df2.columns)
# Find columns that exist only in Dataset 1
only_ds1 = set(df1.columns) - set(df2.columns)
# Find columns that exist only in Dataset 2
only_ds2 = set(df2.columns) - set(df1.columns)

# Print the shared columns and their total number
print(f"Shared columns ({len(shared)}):", sorted(shared))
print()
# Print columns found only in Dataset 1 and their total number
print(f"Dataset 1 only ({len(only_ds1)}):", sorted(only_ds1))
print()
# Print columns found only in Dataset 2 and their total number
print(f"Dataset 2 only ({len(only_ds2)}):", sorted(only_ds2))

Shared columns (19): ['adr', 'arrival_day', 'arrival_month', 'arrival_year', 'is_canceled', 'is_repeated_guest', 'lead_time', 'market_segment', 'meal_plan', 'num_adults', 'num_children', 'previous_bookings_not_canceled', 'previous_cancellations', 'required_car_parking_spaces', 'room_type_reserved', 'source_dataset', 'stays_week_nights', 'stays_weekend_nights', 'total_special_requests']

Dataset 1 only (13): ['agent_id', 'assigned_room_type', 'booking_changes', 'company_id', 'country', 'customer_type', 'days_in_waiting_list', 'deposit_type', 'distribution_channel', 'hotel_type', 'num_babies', 'reservation_status', 'reservation_status_date']

Dataset 2 only (1): ['booking_id']


#Combine Dataset 1 and Dataset 2

In [4]:
# Combine Dataset 1 and Dataset 2 by stacking their rows ,Columns with the same name are matched together
# Missing columns are filled with NaN
df = pd.concat([df1, df2], axis=0, ignore_index=True, sort=False)

# Show the total number of rows and columns after combining
print("Merged shape:", df.shape)
df.head(3)

Merged shape: (155665, 33)


,hotel_type,is_canceled,lead_time,arrival_year,arrival_month,arrival_day,stays_weekend_nights,stays_week_nights,num_adults,num_children,num_babies,meal_plan,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,room_type_reserved,assigned_room_type,booking_changes,deposit_type,agent_id,company_id,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_special_requests,reservation_status,reservation_status_date,source_dataset,booking_id
0,Resort Hotel,0,342,2015,7,1,0,0,2,0,0.0,BB,PRT,Direct,Direct,0,0,0,C,C,3.0,No Deposit,NaN,NaN,0.0,Transient,0.0,0,0,Check-Out,2015-07-01,hotel_bookings_2015_2017,NaN
1,Resort Hotel,0,737,2015,7,1,0,0,2,0,0.0,BB,PRT,Direct,Direct,0,0,0,C,C,4.0,No Deposit,NaN,NaN,0.0,Transient,0.0,0,0,Check-Out,2015-07-01,hotel_bookings_2015_2017,NaN
2,Resort Hotel,0,7,2015,7,1,0,1,1,0,0.0,BB,GBR,Direct,Direct,0,0,0,A,C,0.0,No Deposit,NaN,NaN,0.0,Transient,75.0,0,0,Check-Out,2015-07-02,hotel_bookings_2015_2017,NaN


#Final checks

In [5]:
# row count= df1 + df2
assert df.shape[0] == df1.shape[0] + df2.shape[0]
print("Row count check passed:", df.shape[0], "=", df1.shape[0], "+", df2.shape[0])

# column count = the union
assert df.shape[1] == len(set(df1.columns) | set(df2.columns))
print("Column count check passed:", df.shape[1])

Row count check passed: 155665 = 119390 + 36275
Column count check passed: 33


In [6]:
# Count how many rows came from each dataset
print(df['source_dataset'].value_counts())

source_dataset
hotel_bookings_2015_2017        119390
hotel_reservations_2017_2018     36275
Name: count, dtype: int64


In [7]:
# Check that columns belonging to only one dataset are NaN in the other dataset's rows.
df.groupby('source_dataset')[sorted(only_ds1 | only_ds2)].apply(lambda g: g.isnull().all())

,agent_id,assigned_room_type,booking_changes,booking_id,company_id,country,customer_type,days_in_waiting_list,deposit_type,distribution_channel,hotel_type,num_babies,reservation_status,reservation_status_date
source_dataset,,,,,,,,,,,,,,
hotel_bookings_2015_2017,False,False,False,True,False,False,False,False,False,False,False,False,False,False
hotel_reservations_2017_2018,True,True,True,False,True,True,True,True,True,True,True,True,True,True


#Save merged output

In [8]:
# Set the location where the merged dataset will be saved
OUT_PATH = "data/processed/merged_dataset.csv"
# Save the dataframe as a CSV file without the index
df.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}")

Saved to data/processed/merged_dataset.csv
